In [44]:
%cd /content
!rm -rf ML-final
!git clone https://github.com/ketevan614/ML-final.git
%cd /content/ML-final
!pwd
!ls

/content
Cloning into 'ML-final'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 88 (delta 30), reused 77 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 9.29 MiB | 17.69 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/ML-final
/content/ML-final
data				 model_inference.ipynb
dataloader.py			 models
EXPERIMENTS.md			 neuralforecast_pyfunc.py
extract.py			 notebooks
FEATURES.md			 pipeline.py
features.py			 PROJECT_PLAN.md
metrics.py			 README.md
model_experiment_DLinear.ipynb	 requirements-dl.txt
model_experiment_LightGBM.ipynb  requirements.txt
model_experiment_NBEATS.ipynb	 TASKS.md
model_experiment_SARIMA.ipynb	 validation.py


In [45]:
%cd /content/ML-final
!pwd
!ls data

/content/ML-final
/content/ML-final
walmart-recruiting-store-sales-forecasting
walmart-recruiting-store-sales-forecasting.zip


In [46]:
!git clone https://github.com/ketevan614/ML-final.git
%cd ML-final

Cloning into 'ML-final'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 88 (delta 30), reused 77 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 9.29 MiB | 18.66 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/ML-final/ML-final


In [43]:
!pip install torch "neuralforecast>=1.7,<2" mlflow dagshub pandas numpy scikit-learn pyarrow matplotlib python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.2/259.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 53.1 MB/s eta 0:00:00


In [47]:
!pip install -r requirements.txt

In [48]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_ef32993ba9bf755741a858ec784e7628"


!mkdir -p ~/.kaggle
!echo "$KAGGLE_API_TOKEN" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

In [49]:
import dagshub
dagshub.init(repo_owner="karev23", repo_name="ML-final", mlflow=True)

Initialized MLflow to track repo "karev23/ML-final"

Repository karev23/ML-final initialized!

In [50]:
import mlflow
print(mlflow.get_tracking_uri())


https://dagshub.com/karev23/ML-final.mlflow


In [51]:
!mkdir -p data
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data

!unzip -q data/walmart-recruiting-store-sales-forecasting.zip -d data
!unzip -q data/train.csv.zip -d data
!unzip -q data/test.csv.zip -d data
!unzip -q data/features.csv.zip -d data
!unzip -q data/stores.csv.zip -d data
!unzip -q data/sampleSubmission.csv.zip -d data
!ls data

walmart-recruiting-store-sales-forecasting.zip: Skipping, found more recently modified local copy (use --force to force download)
unzip:  cannot find or open data/stores.csv.zip, data/stores.csv.zip.zip or data/stores.csv.zip.ZIP.
features.csv		  test.csv.zip
features.csv.zip	  train.csv
sampleSubmission.csv	  train.csv.zip
sampleSubmission.csv.zip  walmart-recruiting-store-sales-forecasting
stores.csv		  walmart-recruiting-store-sales-forecasting.zip
test.csv


# DLinear Training

DLinear experiment notebook for the Walmart sales forecasting project using Nixtla `neuralforecast`.

What it does:
- converts Walmart panel data to Nixtla long format: `unique_id`, `ds`, `y`, and exogenous columns;
- trains one global DLinear model across all Store-Dept series;
- evaluates with the shared WMAE metric on 39-week expanding windows;
- logs the required MLflow stages;
- logs a pyfunc wrapper so the final artifact accepts raw test rows.

This notebook depends on the deep-learning environment from `requirements-dl.txt`.

In [52]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dataloader import load_raw, merge_all, make_submission
from features import build_features
from metrics import wmae
from validation import expanding_window_folds

warnings.filterwarnings("ignore")
DATA_DIR = ROOT / "data"
EXPERIMENT_NAME = "DLinear_Training"
MODEL_ALIAS = "DLinear"
HORIZON = 39
RANDOM_STATE = 42

In [53]:
import os
import mlflow

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=True)
except ImportError:
    pass
from neuralforecast import NeuralForecast
from neuralforecast_pyfunc import NeuralForecastRawPyFunc
from neuralforecast.models import DLinear

# Some local environments set a dead proxy (for example 127.0.0.1:9),
# which prevents MLflow from reaching DagsHub. Clear it before tracking calls.
for key in [
    "HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy",
    "ALL_PROXY", "all_proxy",
]:
    os.environ.pop(key, None)

mlflow.set_experiment(EXPERIMENT_NAME)


# Do not let temporary DagsHub/DNS failures crash long training cells.
# Failed log calls are printed; rerun the cell later on a stable connection if a required metric is missing remotely.
def _safe_mlflow_call(fn, label):
    def wrapped(*args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            print(f"[MLflow warning] {label} failed: {type(exc).__name__}: {str(exc)[:300]}")
            return None
    return wrapped

mlflow.log_param = _safe_mlflow_call(mlflow.log_param, "log_param")
mlflow.log_params = _safe_mlflow_call(mlflow.log_params, "log_params")
mlflow.log_metric = _safe_mlflow_call(mlflow.log_metric, "log_metric")
mlflow.log_metrics = _safe_mlflow_call(mlflow.log_metrics, "log_metrics")
mlflow.log_artifact = _safe_mlflow_call(mlflow.log_artifact, "log_artifact")
mlflow.log_artifacts = _safe_mlflow_call(mlflow.log_artifacts, "log_artifacts")

# Run this cell once before the stage cells when you want MLflow to group
# the required Cleaning -> Feature Selection -> CV -> HPO -> Final runs.
parent_run = mlflow.start_run(run_name="DLinear_Workflow")

2026/07/11 22:54:43 INFO mlflow.tracking.fluent: Experiment with name 'DLinear_Training' does not exist. Creating a new experiment.


## Data Preparation

`neuralforecast` expects one row per series and date. `unique_id` identifies a Store-Dept time series. The fallback functions below can be replaced by Gega's shared `nf_prep.py` when it is available.

In [54]:
train_raw, test_raw, features_df, stores_df, sample_submission = load_raw(DATA_DIR)
train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_raw = test_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

def to_nixtla(raw_frame: pd.DataFrame, history_frame: pd.DataFrame | None = None, include_y: bool = True) -> pd.DataFrame:
    merged = merge_all(raw_frame, features_df, stores_df)
    history_merged = None if history_frame is None else merge_all(history_frame, features_df, stores_df)
    X = build_features(merged, sales_history_df=history_merged, encode_categoricals=True)
    nf = pd.DataFrame({
        "unique_id": raw_frame["Store"].astype(str) + "_" + raw_frame["Dept"].astype(str),
        "ds": pd.to_datetime(raw_frame["Date"]),
    })
    if include_y and "Weekly_Sales" in raw_frame.columns:
        nf["y"] = raw_frame["Weekly_Sales"].astype(float).to_numpy()
    for col in X.columns:
        values = X[col]
        if str(values.dtype) == "category":
            values = values.astype(str)
        nf[col] = values.to_numpy()
    return nf

def holiday_lookup(raw_frame: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "unique_id": raw_frame["Store"].astype(str) + "_" + raw_frame["Dept"].astype(str),
        "ds": pd.to_datetime(raw_frame["Date"]),
        "IsHoliday_eval": raw_frame["IsHoliday"].astype(bool).to_numpy(),
    })

## MLflow Stage: Cleaning

Deep models are more sensitive to scale than tree models. The first version keeps negative sales in training but clips final predictions at zero. Sparse or very short series are kept initially; if training fails, filter short series in this stage and log the rule.

In [55]:
with mlflow.start_run(run_name="DLinear_Cleaning", nested=True):
    series_lengths = train_raw.groupby(["Store", "Dept"]).size()
    mlflow.log_param("negative_sales_training_policy", "keep")
    mlflow.log_param("negative_prediction_policy", "clip_to_zero_for_submission")
    mlflow.log_param("short_series_policy", "keep_initially")
    mlflow.log_metric("series_count", int(series_lengths.shape[0]))
    mlflow.log_metric("min_series_length", int(series_lengths.min()))
    mlflow.log_metric("median_series_length", float(series_lengths.median()))

🏃 View run DLinear_Cleaning at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/4/runs/71d1999a1a34471497788b712792cde5
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/4


## MLflow Stage: Feature Selection

DLinear's core strength is decomposing each series into trend and seasonality, so the first feature set is conservative: calendar, holiday, store, markdown, external variables, and safe history features from `features.py`.

In [56]:
nf_train_preview = to_nixtla(train_raw)
BASE_COLS = ["unique_id", "ds", "y"]
ALL_EXOG_FEATURES = [c for c in nf_train_preview.columns if c not in BASE_COLS]
FUTR_EXOG_LIST = []

with mlflow.start_run(run_name="DLinear_Feature_Selection", nested=True):
    mlflow.log_param("feature_source", "shared_features_py_encoded_for_neuralforecast")
    mlflow.log_param("uses_future_exog", True)
    mlflow.log_metric("future_exog_count", len(FUTR_EXOG_LIST))
    pd.Series(FUTR_EXOG_LIST, name="feature").to_csv(ROOT / "models" / "dlinear_exog_features.csv", index=False)
    mlflow.log_artifact(str(ROOT / "models" / "dlinear_exog_features.csv"))

🏃 View run DLinear_Feature_Selection at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/4/runs/34893dbfa3e049cab99ee40d93bab8be
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/4


## Cross-Validation Helpers

Each fold trains a global DLinear model on all series up to the cutoff and predicts the next 39 dates. WMAE is computed after merging predictions back to the validation rows.

In [57]:
def build_dlinear(input_size: int = 104, max_steps: int = 500):
    return DLinear(
        h=HORIZON,
        input_size=input_size,
        max_steps=max_steps,
        learning_rate=1e-3,
        futr_exog_list=FUTR_EXOG_LIST,
        alias=MODEL_ALIAS,
        random_seed=RANDOM_STATE,
    )

def cv_dlinear(input_size: int = 104, max_steps: int = 500) -> list[float]:
    scores = []
    for fold, (tr_mask, val_mask) in enumerate(expanding_window_folds(train_raw["Date"], n_splits=3, horizon=HORIZON), start=1):
        fold_train_raw = train_raw.loc[tr_mask].copy()
        fold_val_raw = train_raw.loc[val_mask].copy()
        nf_train = to_nixtla(fold_train_raw)
        nf_val = to_nixtla(fold_val_raw, history_frame=fold_train_raw)
        nf = NeuralForecast(models=[build_dlinear(input_size=input_size, max_steps=max_steps)], freq="W-FRI")
        nf.fit(df=nf_train)
        preds = nf.predict(futr_df=nf_val.drop(columns=["y"]))
        scored = nf_val[["unique_id", "ds", "y"]].merge(preds[["unique_id", "ds", MODEL_ALIAS]], on=["unique_id", "ds"], how="left")
        scored = scored.merge(holiday_lookup(fold_val_raw), on=["unique_id", "ds"], how="left")
        pred = np.clip(scored[MODEL_ALIAS].to_numpy(), 0, None)
        scores.append(wmae(scored["y"], pred, scored["IsHoliday_eval"]))
    return scores

In [58]:
with mlflow.start_run(run_name="DLinear_CrossValidation", nested=True):
    cv_scores = cv_dlinear(input_size=104, max_steps=500)
    for i, score in enumerate(cv_scores, start=1):
        mlflow.log_metric(f"fold_{i}_wmae", score)
    mlflow.log_metric("mean_cv_wmae", float(np.mean(cv_scores)))

INFO:lightning_fabric.utilities.seed:Seed set to 42


🏃 View run DLinear_CrossValidation at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/4/runs/6536feff7b46488fab93bbbb30ac9fdc
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/4


Exception: DLinear does not support future exogenous variables.

## MLflow Stage: HPO

Keep DLinear tuning small. The useful knobs are mostly `input_size`, learning rate, and training steps. Increase only after the baseline run is stable.

In [ ]:
SEARCH_SPACE = [
    {"input_size": 52, "max_steps": 500},
    {"input_size": 104, "max_steps": 500},
    {"input_size": 104, "max_steps": 1000},
]

with mlflow.start_run(run_name="DLinear_HPO", nested=True):
    results = []
    for params in SEARCH_SPACE:
        scores = cv_dlinear(**params)
        results.append({**params, "mean_wmae": float(np.mean(scores))})
    hpo_results = pd.DataFrame(results).sort_values("mean_wmae")
    best_params = hpo_results.iloc[0].drop("mean_wmae").to_dict()
    hpo_results.to_csv(ROOT / "models" / "dlinear_hpo_results.csv", index=False)
    mlflow.log_params(best_params)
    mlflow.log_metric("best_mean_cv_wmae", float(hpo_results.iloc[0]["mean_wmae"]))
    mlflow.log_artifact(str(ROOT / "models" / "dlinear_hpo_results.csv"))

## MLflow Stage: Final Model

The final training uses all training rows, creates the raw test submission, and logs an `mlflow.pyfunc.PythonModel` wrapper so inference can start from raw test rows.

In [ ]:
with mlflow.start_run(run_name="DLinear_Final", nested=True):
    final_params = globals().get("best_params", {"input_size": 104, "max_steps": 500})
    nf_train = to_nixtla(train_raw)
    nf_test = to_nixtla(test_raw, history_frame=train_raw, include_y=False)
    final_nf = NeuralForecast(models=[build_dlinear(**final_params)], freq="W-FRI")
    final_nf.fit(df=nf_train)

    test_preds = final_nf.predict(futr_df=nf_test)
    test_pred_values = np.clip(test_preds[MODEL_ALIAS].to_numpy(), 0, None)
    make_submission(test_raw, test_pred_values, ROOT / "models" / "dlinear_submission.csv")

    mlflow.log_params(final_params)
    mlflow.log_artifact(str(ROOT / "models" / "dlinear_submission.csv"))
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=NeuralForecastRawPyFunc(
            neuralforecast_model=final_nf,
            features_df=features_df,
            stores_df=stores_df,
            train_history_raw=train_raw,
            model_alias=MODEL_ALIAS,
            uses_exog=True,
        ),
        code_paths=[
            str(ROOT / "neuralforecast_pyfunc.py"),
            str(ROOT / "dataloader.py"),
            str(ROOT / "features.py"),
        ],
        pip_requirements=[
            "mlflow",
            "numpy",
            "pandas",
            "scikit-learn",
            "torch",
            "neuralforecast>=1.7,<2",
        ],
    )

mlflow.end_run()


## README Takeaways

- DLinear is a fast deep-learning baseline that decomposes each series into linear trend and seasonal components.
- It is useful as a complexity check: if it is close to N-BEATS, the extra neural complexity may not be worth it.
- Compare DLinear against LightGBM especially on holiday weeks, because DLinear may smooth sharp holiday spikes.